In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense
from sklearn.model_selection import train_test_split

In [4]:
texts = [
    "win lottery now",
    "claim your free money",
    "lottery winner announced",
    "meeting at office tomorrow",
    "project discussion with team",
    "family dinner at home"
]

labels = ['spam', 'spam', 'spam', 'ham', 'ham', 'ham']

In [5]:
df = pd.DataFrame({'text': texts, 'label': labels})
df.head()

,text,label
0,win lottery now,spam
1,claim your free money,spam
2,lottery winner announced,spam
3,meeting at office tomorrow,ham
4,project discussion with team,ham


In [6]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])
df.head()

,text,label
0,win lottery now,1
1,claim your free money,1
2,lottery winner announced,1
3,meeting at office tomorrow,0
4,project discussion with team,0


In [7]:
y = df['label'].values

In [8]:
y

array([1, 1, 1, 0, 0, 0])

In [9]:
tokeniser = Tokenizer(num_words=1000, oov_token="<OOV>") # Keep 1000 words in memoery and put <OOV> for any out of vocabulary word
tokeniser.fit_on_texts(df['text'])


In [10]:
tokeniser

In [11]:
X_seq = tokeniser.texts_to_sequences(df['text'])
print("Sequences ", X_seq)

Sequences  [[4, 2, 5], [6, 7, 8, 9], [2, 10, 11], [12, 3, 13, 14], [15, 16, 17, 18], [19, 20, 3, 21]]


In [12]:
max_len = max([len(seq) for seq in X_seq]) # List comprehension
print("Max length: ", max_len)

Max length:  4


In [14]:
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post') # so that the length is same for all the elements. Here it's 4
print(X_pad)

[[ 4  2  5  0]
 [ 6  7  8  9]
 [ 2 10 11  0]
 [12  3 13 14]
 [15 16 17 18]
 [19 20  3 21]]


In [15]:
vocab_size = len(tokeniser.word_index) + 1
print("Vocab size: ", vocab_size)

Vocab size:  22


Total words we got is 22.

In [16]:
embedding_dim = 16

In [17]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)) # Number of features
model.add(GRU(units=32))
model.add(Dense(units=1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy']) # As this is a classification we try to find accuracy hence metrics is accuracy

In [33]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (2, 4, 16)             │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (2, 32)                │         4,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (2, 1)                 │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,557 (60.77 KB)

 Trainable params: 5,185 (20.25 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 10,372 (40.52 KB)

In [22]:
history = model.fit(X_pad, y, epochs=20, batch_size=2)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 1.0000 - loss: 0.0510
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 1.0000 - loss: 0.0527
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.0487
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 1.0000 - loss: 0.0357 
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 1.0000 - loss: 0.0401
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.0266
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 1.0000 - loss: 0.0274
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 1.0000 - loss: 0.0202
Epoch 9/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 1.0000 - loss: 0.0196
Epoch 10/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 1.0000 - loss: 0.0168
Epoch 11/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 1.0000 - loss: 0.0170
Epoch 12/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.0110


In [28]:
test_texts = [
    "free lottery now",
    "meeting with project team"
]

In [29]:
test_seq = tokeniser.texts_to_sequences(test_texts) # tokenize the test data
test_pad = pad_sequences(test_seq, maxlen=max_len, padding='post') # add padding
pred_probs = model.predict(test_pad)
print(pred_probs)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
[[0.9985557 ]
 [0.01005525]]


greater than 0.5 (0.9985557) it's a spam and the second one is a ham as it is less than 0.5

In [27]:
# Inverse classify
pred_labels = (pred_probs > 0.5).astype(int).flatten()
print(pred_labels)

[1 0]


In [32]:
pred_class_names = label_encoder.inverse_transform(pred_labels)
for t,c,p in zip(test_texts, pred_class_names, pred_probs):
    print(f"Text: {t!r} -> Predicted: {c} (Prob = {float(p): .3f})")

Text: 'free lottery now' -> Predicted: spam (Prob =  0.999)
Text: 'meeting with project team' -> Predicted: ham (Prob =  0.010)


/tmp/ipython-input-113711824.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"Text: {t!r} -> Predicted: {c} (Prob = {float(p): .3f})")
